# Evaluación de NV-Reason-CT en COVID-19-CT-Seg

Este notebook consulta `nvidia/NV-Reason-CT` sobre todos los volúmenes CT encontrados en el directorio configurado. El dataset [COVID-19 CT Lung and Infection Segmentation](https://zenodo.org/records/3757476) contiene 20 CT etiquetadas como COVID-19 positivas y máscaras de pulmón/infección. Se espera encontrar archivos `volume-covid19-*_ct.nii.gz` (también se acepta `.nii`).

NV-Reason-CT es un modelo visión-lenguaje 3D: espera CT NIfTI en unidades Hounsfield y genera respuestas de texto, no máscaras. Por eso este flujo **no calcula Dice**. Como todos los casos son positivos, tampoco puede estimar especificidad ni precisión positiva; reportará sensibilidad descriptiva entre respuestas binarias y cobertura. El posible solapamiento con los datos de entrenamiento del modelo no está verificado.

Uso educativo y de investigación; no sirve para diagnóstico ni decisiones clínicas.

## 1. Configurar entorno y dependencias

Ejecuta este notebook exclusivamente en un servidor Linux con GPU NVIDIA CUDA y Python 3.11 o superior. El modelo tiene aproximadamente 5 mil millones de parámetros; el model card documenta pruebas con H100/L40S. Acepta y revisa la licencia OpenMDW-1.1.

Prepara el entorno en el servidor antes de abrir el notebook. Esta instrucción es texto, no una celda ejecutable:

```bash
python -m pip install -r https://huggingface.co/nvidia/NV-Reason-CT/resolve/main/requirements.txt
```

No instales dependencias, descargues pesos ni ejecutes el notebook en el equipo local. [Model card de NV-Reason-CT](https://huggingface.co/nvidia/NV-Reason-CT).

## 2. Definir parámetros y rutas de trabajo

Configura `COVID_CT_TRAIN_DIR` en el servidor o edita la ruta por defecto de la celda siguiente. `COVID_CT_RESULTS_CSV` permite cambiar la ubicación del CSV de resultados.

In [ ]:
import logging
import os
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("nv_reason_ct_covid")

DATA_DIR = Path(os.environ.get(
    "COVID_CT_TRAIN_DIR",
    "/ruta/en/servidor/COVID-19-CT-Seg_20cases",
))
OUTPUT_CSV = Path(os.environ.get(
    "COVID_CT_RESULTS_CSV",
    str(DATA_DIR / "nv_reason_ct_train_predictions.csv"),
))
EXPECTED_CASES = 20
RESUME = True
MODEL_ID = "nvidia/NV-Reason-CT"
PROMPT_VERSION = "covid_binary_v1"

## 3. Crear estructuras base de datos/entrada

La celda siguiente busca recursivamente los volúmenes NIfTI del directorio indicado, ignora máscaras y otros archivos, y avisa si el número encontrado difiere de los 20 casos esperados. Revisa la lista impresa para confirmar que corresponde al split que quieres evaluar.

In [ ]:
import re
from collections import Counter

ct_filename_pattern = re.compile(r"^volume-covid19-.*_ct\.nii(?:\.gz)?$", re.IGNORECASE)
image_paths = sorted(
    path for path in DATA_DIR.rglob("*")
    if path.is_file() and ct_filename_pattern.match(path.name)
)
if not image_paths:
    raise FileNotFoundError(
        f"No se encontraron CT con patrón volume-covid19-*_ct.nii[.gz] en {DATA_DIR}."
    )

basename_counts = Counter(path.name for path in image_paths)
duplicate_basenames = sorted(name for name, count in basename_counts.items() if count > 1)
if duplicate_basenames:
    raise ValueError(f"Nombres de archivo duplicados: {duplicate_basenames}")
if len(image_paths) != EXPECTED_CASES:
    logger.warning(
        "Se esperaban %d volúmenes y se encontraron %d; confirma que DATA_DIR "
        "apunte al split previsto.",
        EXPECTED_CASES,
        len(image_paths),
    )

logger.info("Volúmenes CT encontrados: %d", len(image_paths))
for image_path in image_paths:
    logger.info("Entrada: %s", image_path)

In [ ]:
import sys
import torch

if sys.version_info < (3, 11):
    raise RuntimeError("Se requiere Python 3.11 o superior en el servidor.")
if not torch.cuda.is_available():
    raise RuntimeError(
        "No se detectó CUDA. Este notebook requiere el servidor Linux con GPU NVIDIA; "
        "no se usa fallback local a CPU o MPS."
    )

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 4. Implementar funciones núcleo

La función de parseo acepta una respuesta que empiece explícitamente por `YES` o `NO`. Cualquier otra formulación queda como ambigua; no se fuerza una predicción a partir del razonamiento libre del modelo.

In [ ]:
import re

PROMPT = (
    "Does this chest CT show pulmonary findings compatible with COVID-19 "
    "pneumonia? Answer only YES or NO."
)


def parse_binary_answer(text):
    """Acepta solo YES/NO al inicio; conserva otros formatos como ambiguos."""
    first_line = text.strip().splitlines()[0] if text.strip() else ""
    first_line = re.sub(r"^[\s*_`#>\-]+", "", first_line)
    match = re.match(r"(?i)(?:answer\s*:\s*)?(yes|no)\b", first_line)
    return match.group(1).lower() if match else ""


assert parse_binary_answer("YES") == "yes"
assert parse_binary_answer("**No.**") == "no"
assert parse_binary_answer("The findings may be compatible: yes") == ""
print("Pruebas rápidas del parser: correctas")

## 5. Ejecutar flujo principal

La celda siguiente carga los pesos y el processor según el model card. Después, la inferencia procesa cada volumen secuencialmente, con `do_sample=False` y sin submuestreo. El checkpoint conserva salidas por caso para auditar o reanudar.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
).eval().to("cuda")

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

print(f"Modelo cargado en {model.device}: {MODEL_ID}")

In [ ]:
import csv
import time

RESULT_FIELDS = [
    "case_id",
    "image_path",
    "model_id",
    "prompt_version",
    "prompt",
    "raw_response",
    "prediction",
    "status",
    "error",
    "elapsed_seconds",
]


def read_completed_checkpoint():
    """Lee respuestas ya obtenidas para el mismo modelo y versión de prompt."""
    if not RESUME or not OUTPUT_CSV.exists():
        return {}

    with OUTPUT_CSV.open(newline="", encoding="utf-8") as result_file:
        previous_rows = csv.DictReader(result_file)
        return {
            row["case_id"]: row
            for row in previous_rows
            if row.get("model_id") == MODEL_ID
            and row.get("prompt_version") == PROMPT_VERSION
            and row.get("status") in {"ok", "ambiguous"}
            and (
                row.get("status") == "ambiguous"
                or row.get("prediction") in {"yes", "no"}
            )
        }


def write_checkpoint(rows_by_case):
    """Escribe el estado completo mediante reemplazo atómico del CSV."""
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = OUTPUT_CSV.with_suffix(OUTPUT_CSV.suffix + ".tmp")
    ordered_rows = [rows_by_case[key] for key in sorted(rows_by_case)]
    with temporary_path.open("w", newline="", encoding="utf-8") as result_file:
        writer = csv.DictWriter(result_file, fieldnames=RESULT_FIELDS)
        writer.writeheader()
        writer.writerows(ordered_rows)
    temporary_path.replace(OUTPUT_CSV)


results_by_case = read_completed_checkpoint()
current_paths_by_case = {path.name: str(path) for path in image_paths}
results_by_case = {
    case_id: row for case_id, row in results_by_case.items()
    if current_paths_by_case.get(case_id) == row.get("image_path")
}
print(f"Respuestas previas recuperadas: {len(results_by_case)}")
print(f"CSV de resultados: {OUTPUT_CSV}")

for case_number, image_path in enumerate(image_paths, start=1):
    case_id = image_path.name
    if case_id in results_by_case:
        print(f"[{case_number}/{len(image_paths)}] Reutilizado: {case_id}")
        continue

    started_at = time.perf_counter()
    raw_response = ""
    prediction = ""
    status = "error"
    error_message = ""

    try:
        messages = [{
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": PROMPT},
            ],
        }]
        formatted_prompt = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = processor(
            text=formatted_prompt,
            images3d=[str(image_path)],
            anatomy_region="chest",
            return_tensors="pt",
        ).to(model.device)

        with torch.inference_mode():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=32,
                do_sample=False,
                use_cache=True,
            )

        new_tokens = generated_ids[:, inputs.input_ids.shape[1]:]
        raw_response = processor.batch_decode(
            new_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0].strip()
        prediction = parse_binary_answer(raw_response)
        status = "ok" if prediction in {"yes", "no"} else "ambiguous"
    except (RuntimeError, ValueError, OSError, TypeError, KeyError, IndexError) as error:
        error_message = f"{type(error).__name__}: {error}"

    result_row = {
        "case_id": case_id,
        "image_path": str(image_path),
        "model_id": MODEL_ID,
        "prompt_version": PROMPT_VERSION,
        "prompt": PROMPT,
        "raw_response": raw_response,
        "prediction": prediction,
        "status": status,
        "error": error_message,
        "elapsed_seconds": f"{time.perf_counter() - started_at:.3f}",
    }
    results_by_case[case_id] = result_row
    write_checkpoint(results_by_case)

    if status == "error":
        print(f"[{case_number}/{len(image_paths)}] Error en {case_id}: {error_message}")
    else:
        print(
            f"[{case_number}/{len(image_paths)}] {case_id}: "
            f"{prediction or 'ambigua'} ({result_row['elapsed_seconds']} s)"
        )

print(f"Inferencia terminada. Resultados guardados en: {OUTPUT_CSV}")

## 6. Validar con pruebas rápidas

Estas comprobaciones son del flujo, no ejecutan el modelo: verifican que haya una fila por volumen y que cada estado tenga una predicción coherente. Las aserciones del parser en la sección 4 cubren respuestas sí, no y ambiguas.

In [ ]:
expected_case_ids = {image_path.name for image_path in image_paths}
recorded_case_ids = set(results_by_case)
assert expected_case_ids.issubset(recorded_case_ids), "Faltan casos en el checkpoint."
assert all(
    row["status"] in {"ok", "ambiguous", "error"}
    for row in results_by_case.values()
), "Hay estados de resultado desconocidos."
assert all(
    row["prediction"] in {"yes", "no"}
    for row in results_by_case.values()
    if row["status"] == "ok"
), "Una respuesta marcada como ok no tiene clase binaria."
assert all(
    not row["prediction"]
    for row in results_by_case.values()
    if row["status"] != "ok"
), "Una respuesta ambigua/error no debe tener clase binaria."
print(f"Checkpoint coherente: {len(expected_case_ids)} casos del directorio registrados")

## 7. Registrar resultados y artefactos

El CSV conserva por volumen la respuesta textual, la clase parseada, el estado, el prompt y el tiempo de inferencia. Las métricas se calculan desde el CSV para que el resultado sea auditable y reproducible.

La sensibilidad se calcula entre los casos con respuesta binaria. La proporción de respuestas afirmativas sobre todos los casos incluye ambiguos y errores como no detectados, y se reporta aparte. Ninguno de estos valores permite estimar especificidad ni rendimiento clínico; el modelo no genera máscaras y no se compara con las anotaciones de infección.

In [ ]:
with OUTPUT_CSV.open(newline="", encoding="utf-8") as result_file:
    all_rows = list(csv.DictReader(result_file))

rows_by_case = {row["case_id"]: row for row in all_rows}
case_rows = [rows_by_case[path.name] for path in image_paths if path.name in rows_by_case]
binary_rows = [row for row in case_rows if row["prediction"] in {"yes", "no"}]
ambiguous_count = sum(row["status"] == "ambiguous" for row in case_rows)
error_count = sum(row["status"] == "error" for row in case_rows)
true_positive = sum(row["prediction"] == "yes" for row in binary_rows)
false_negative = sum(row["prediction"] == "no" for row in binary_rows)
total_cases = len(image_paths)
binary_count = len(binary_rows)

sensitivity_valid = true_positive / binary_count if binary_count else None
coverage = binary_count / total_cases if total_cases else None
positive_response_rate_all = true_positive / total_cases if total_cases else None

print(f"CT encontrados: {total_cases}")
print(f"Respuestas binarias válidas: {binary_count}")
print(f"Respuestas ambiguas: {ambiguous_count}")
print(f"Errores de inferencia: {error_count}")
print(f"Sí detectado (TP): {true_positive}")
print(f"No detectado (FN): {false_negative}")
print(f"Cobertura binaria: {coverage:.1%}" if coverage is not None else "Cobertura binaria: NA")
print(
    "Sensibilidad descriptiva entre respuestas binarias: "
    f"{sensitivity_valid:.1%}" if sensitivity_valid is not None
    else "Sensibilidad descriptiva entre respuestas binarias: NA"
)
print(
    "Proporción de todos los casos con respuesta sí "
    f"(ambigüedades/errores no detectados): {positive_response_rate_all:.1%}"
    if positive_response_rate_all is not None
    else "Proporción de respuestas sí: NA"
)
print("Especificidad: no estimable; no hay casos negativos en esta cohorte.")
print("Precisión positiva: no estimable; no hay casos negativos para contar falsos positivos.")

print("\nResumen por caso:")
for row in case_rows:
    print(
        f"{row['case_id']}: predicción={row['prediction'] or 'NA'} | "
        f"estado={row['status']} | {row['raw_response'][:180]}"
    )